# Inspect a Random 64D AlphaEarth Pixel

This notebook reads the AlphaEarth mosaic and prints the 64-band embedding values at a random valid pixel.

In [1]:
import rasterio
from pyproj import Transformer
from pathlib import Path
import numpy as np

def find_repo_root(start: Path) -> Path | None:
    for candidate in [start] + list(start.parents):
        if (candidate / "data").exists() and (candidate / "notebooks").exists():
            return candidate
    return None

repo_root = find_repo_root(Path.cwd())
if repo_root is None:
    raise FileNotFoundError("Could not locate repo root containing data/ and notebooks/.")

raster_path = repo_root / "data/interim/bd_coastal_alphaearth_2024_mosaic.tif"

if not raster_path.exists():
    candidates = sorted((repo_root / "data/interim").glob("*alphaearth*.tif"))
    message = [
        f"Mosaic file not found at {raster_path}.",
        "Run scripts/gee/mosaic_alphaearth_tiles.py or update raster_path.",
    ]
    if candidates:
        message.append("Found these candidates under data/interim:\n  - " + "\n  - ".join(str(p) for p in candidates))
    raise FileNotFoundError("\n".join(message))

raster_path


PosixPath('/mnt/AAzizSSD/BDlulcModel/data/interim/bd_coastal_alphaearth_2024_mosaic.tif')

In [2]:
with rasterio.open(raster_path) as src:
    src_crs = src.crs
    if src_crs is None:
        raise ValueError("Raster CRS is undefined")

    nodata = src.nodata
    if nodata is None:
        nodata = 0.0

    # Find a random valid pixel using band 1 (downsampled for speed).
    max_dim = 1200
    scale = max(src.width / max_dim, src.height / max_dim, 1)
    out_width = max(1, int(src.width / scale))
    out_height = max(1, int(src.height / scale))

    band1_preview = src.read(
        1,
        out_shape=(out_height, out_width),
        resampling=rasterio.enums.Resampling.nearest,
    )
    valid_mask = np.isfinite(band1_preview) & (band1_preview != nodata)
    if not valid_mask.any():
        raise ValueError("No valid pixels found (all nodata).")

    rows, cols = np.where(valid_mask)
    idx = np.random.randint(0, len(rows))
    row_preview, col_preview = int(rows[idx]), int(cols[idx])

    row = int(row_preview * scale)
    col = int(col_preview * scale)

    x, y = src.xy(row, col)
    if src_crs.to_string() != "EPSG:4326":
        transformer = Transformer.from_crs(src_crs, "EPSG:4326", always_xy=True)
        lon, lat = transformer.transform(x, y)
    else:
        lon, lat = x, y

    # Read all bands at the chosen pixel.
    window = ((row, row + 1), (col, col + 1))
    sample = src.read(window=window).reshape(src.count)

print(f"Random valid pixel row={row}, col={col}")
print(f"Lon, Lat: {lon:.6f}, {lat:.6f}")
print(f"Bands: {sample.shape[0]}")
for i, v in enumerate(sample.astype(np.float64), start=1):
    print(f"Band {i:02d}: {v:.16f}")


Random valid pixel row=8742, col=10734
Lon, Lat: 89.430565, 22.964577
Bands: 64
Band 01: 0.0888273740868896
Band 02: -0.0842137639369473
Band 03: 0.0711111111111111
Band 04: 0.0222068435217224
Band 05: -0.0553633217993080
Band 06: 0.0984236831987697
Band 07: -0.1190926566705113
Band 08: 0.0103960015378700
Band 09: -0.1190926566705113
Band 10: 0.2679584775086506
Band 11: 0.2069357939254133
Band 12: -0.0629911572472126
Band 13: -0.3014225297962323
Band 14: -0.1137408688965783
Band 15: -0.0797231833910035
Band 16: 0.1476970396001538
Band 17: -0.1190926566705113
Band 18: -0.1245674740484429
Band 19: -0.1537870049980777
Band 20: 0.0199307958477509
Band 21: -0.1929104190695886
Band 22: 0.2441522491349481
Band 23: 0.0384467512495194
Band 24: 0.0482276047673972
Band 25: 0.0591157247212611
Band 26: 0.2519646289888504
Band 27: 0.0517339484813533
Band 28: 0.0297731641676278
Band 29: -0.0120569011918493
Band 30: 0.1301653210303729
Band 31: 0.1085121107266436
Band 32: -0.0297731641676278
Band 33: -